In [ ]:
import nbformat
import os

notebook_name = "Day19AgenticAI.ipynb" # This is the file the user wants to fix

if not os.path.exists(notebook_name):
    print(f"Error: The notebook file '{notebook_name}' was not found.")
    print("Please upload the notebook file you wish to fix to your Colab environment or provide the correct path.")
else:
    with open(notebook_name, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    # Remove widget metadata
    if "widgets" in nb["metadata"]:
        del nb["metadata"]["widgets"]

    with open("fixed_notebook.ipynb", "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    print("Fixed notebook saved to 'fixed_notebook.ipynb'!")

Error: The notebook file 'Day19AgenticAI.ipynb' was not found.
Please upload the notebook file you wish to fix to your Colab environment or provide the correct path.


In [5]:
# Scenario-Based Teaching: Smart Travel Agent
# 🧑‍🏫 How You Start in Class

# Say this:

# “Imagine you are very busy and you tell your assistant:

# ‘Find me the best flight from Delhi to Mumbai.’

# Now instead of replying with just text, your assistant actually thinks,
# checks data, and gives a final decision.”


# Step 1 — Define Tools

# 👉  “Tools = functions the agent can call”

def search_flights(source, destination):
    return [
        {"airline": "IndiGo", "price": 4500, "time": "6:00 AM"},
        {"airline": "Air India", "price": 5200, "time": "9:00 AM"}
    ]

def check_weather(city):
    return {"city": city, "temperature": 28, "condition": "Clear"}

def calculate_budget(flights):
    cheapest = min(flights, key=lambda x: x["price"])
    return cheapest

In [6]:
# Step 2 — Agent Brain (Decision + Planning)
def agent_brain(user_query):
    plan = []

    if "flight" in user_query.lower():
        plan.append("search_flights")
        plan.append("check_weather")
        plan.append("choose_best")

    return plan

In [7]:
# Step 3 — Execution Engine
def execute_plan(plan):
    context = {}

    for step in plan:
        if step == "search_flights":
            print("\n[Action] Searching Flights...")
            context["flights"] = search_flights("Delhi", "Mumbai")
            print(context["flights"])

        elif step == "check_weather":
            print("\n[Action] Checking Weather...")
            context["weather"] = check_weather("Mumbai")
            print(context["weather"])

        elif step == "choose_best":
            print("\n[Action] Choosing Best Option...")
            best = calculate_budget(context["flights"])
            context["best_flight"] = best
            print(best)

    return context

In [8]:
# Step 4 — Full Agent
def travel_agent(user_query):
    print("User Query:", user_query)

    # Step 1: Plan
    plan = agent_brain(user_query)
    print("\n[Plan]", plan)

    # Step 2: Execute
    result = execute_plan(plan)

    # Step 3: Final Response
    final_output = f"""
Best Flight: {result['best_flight']['airline']} at ₹{result['best_flight']['price']}
Weather: {result['weather']['condition']}, {result['weather']['temperature']}°C
Recommendation: Book the cheapest early flight.
"""
    return final_output

In [9]:
# Step 5 — Run It
response = travel_agent("Find best flight from Delhi to Mumbai")
print("\nFinal Answer:\n", response)

User Query: Find best flight from Delhi to Mumbai

[Plan] ['search_flights', 'check_weather', 'choose_best']

[Action] Searching Flights...
[{'airline': 'IndiGo', 'price': 4500, 'time': '6:00 AM'}, {'airline': 'Air India', 'price': 5200, 'time': '9:00 AM'}]

[Action] Checking Weather...
{'city': 'Mumbai', 'temperature': 28, 'condition': 'Clear'}

[Action] Choosing Best Option...
{'airline': 'IndiGo', 'price': 4500, 'time': '6:00 AM'}

Final Answer:
 
Best Flight: IndiGo at ₹4500
Weather: Clear, 28°C
Recommendation: Book the cheapest early flight.



In [10]:
!pip install requests

In [11]:
# Upgrade: Add Reasoning (VERY IMPORTANT)

def choose_best_flight(flights, weather):
    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Choose {cheapest['airline']} at ₹{cheapest['price']}"

In [12]:
# Fraud Detection ReAct Agent (toy demo)
# Scenario: The Bank’s AI Fraud Agent
# Imagine a bank has deployed an AI-powered fraud detection agent named FraudLLM.
# Its job is to monitor transactions in real time and decide whether they are normal or suspicious.
# Step 1: Thinking (Reasoning)
# Whenever a transaction comes in, FraudLLM first analyzes the details:
# - Example: “Payment to overseas account” → FraudLLM thinks: “Analyzing transaction:
# Payment to overseas account”
# Step 2: Planning (Choosing an Action)
# Based on the thought, FraudLLM decides which tool to use:
# - If the transaction involves overseas transfers or multiple small deposits, it plans to flag as suspicious.
# - Otherwise, it plans to mark as normal.
# Step 3: Acting (Running Tools)
# FraudLLM hands over the decision to FraudTools, which executes the action:
# - flag_suspicious → returns “Suspicious Transaction”
# - mark_normal → returns “Normal Transaction”
# Step 4: Answering (Final Decision)
# Once a clear classification is reached, FraudLLM announces the decision:
# - “Decision: Suspicious Transaction”
# - “Decision: Normal Transaction”


class FraudLLM:
    def think(self, input_text):
        # Simulate reasoning
        return f"Analyzing transaction: {input_text}"

    def plan(self, thought):
        # Decide which tool to use
        if "overseas" in thought or "multiple deposits" in thought:
            return "flag_suspicious"
        else:
            return "mark_normal"

    def answer(self, obs):
        # Final answer
        return f"Decision: {obs}"

class FraudTools:
    def run(self, action):
        # Execute fraud detection actions
        if action == "flag_suspicious":
            return "Suspicious Transaction"
        elif action == "mark_normal":
            return "Normal Transaction"
        else:
            return "Unknown"

def is_final(obs):
    # Stop once we have a clear classification
    return "Transaction" in obs

# Instantiate components
llm = FraudLLM()
tools = FraudTools()

def react_agent(transaction):
    thought = llm.think(transaction)
    action = llm.plan(thought)
    obs = tools.run(action)

    # Loop until final decision
    while not is_final(obs):
        thought = llm.think(obs)
        action = llm.plan(thought)
        obs = tools.run(action)

    return llm.answer(obs)

# 🚀 Try it out
print(react_agent("Payment to overseas account"))
print(react_agent("Salary credited to employee account"))
print(react_agent("Multiple small deposits from different sources"))


Decision: Suspicious Transaction
Decision: Normal Transaction
Decision: Normal Transaction


In [13]:
# Fraud Detection ReAct Agent (toy demo)

class FraudLLM:
    def think(self, input_text):
        # Simulate reasoning
        return f"Analyzing transaction: {input_text}"

    def plan(self, thought):
        # Decide which tool to use
        if "overseas" in thought or "multiple deposits" in thought:
            return "flag_suspicious"
        else:
            return "mark_normal"

    def answer(self, obs):
        # Final answer
        return f"Decision: {obs}"

class FraudTools:
    def run(self, action):
        # Execute fraud detection actions
        if action == "flag_suspicious":
            return "Suspicious Transaction"
        elif action == "mark_normal":
            return "Normal Transaction"
        else:
            return "Unknown"

def is_final(obs):
    # Stop once we have a clear classification
    return "Transaction" in obs

# Instantiate components
llm = FraudLLM()
tools = FraudTools()

def react_agent(transaction):
    thought = llm.think(transaction)
    action = llm.plan(thought)
    obs = tools.run(action)

    # Loop until final decision
    while not is_final(obs):
        thought = llm.think(obs)
        action = llm.plan(thought)
        obs = tools.run(action)

    return llm.answer(obs)

# 🚀 Try it out
print(react_agent("Payment to overseas account"))
print(react_agent("Salary credited to employee account"))
print(react_agent("Multiple small deposits from different sources"))

Decision: Suspicious Transaction
Decision: Normal Transaction
Decision: Normal Transaction


senario 2
Scenario: Smart Shopping Assistant 🛒
You say to your students:
“Imagine you are planning a party and you tell your assistant:
‘Find me the best chocolate cake under ₹500.’
Now instead of just replying with text, your assistant actually thinks, checks different bakery websites, compares prices, and then gives you the final decision.”

In [14]:
# Smart Shopping Assistant 🛒

# Step 1: Sample data (pretend this came from websites/APIs)
cakes = [
    {"name": "Chocolate Truffle Cake", "price": 450, "rating": 4.5, "shop": "Sweet Treats"},
    {"name": "Dark Chocolate Cake", "price": 550, "rating": 4.7, "shop": "Cake World"},
    {"name": "Chocolate Fudge Cake", "price": 400, "rating": 4.3, "shop": "Bake House"},
    {"name": "Premium Chocolate Cake", "price": 500, "rating": 4.8, "shop": "Delight Bakery"},
    {"name": "Budget Chocolate Cake", "price": 300, "rating": 4.0, "shop": "Local Bakers"}
]


# Step 2: Assistant function
def find_best_cake(budget):
    print(" Searching for chocolate cakes under ₹", budget)

    # Step 3: Filter cakes within budget
    filtered = [cake for cake in cakes if cake["price"] <= budget]

    if not filtered:
        return "f No cakes found within your budget."

    # Step 4: Sort by rating (highest first)
    best_cake = sorted(filtered, key=lambda x: x["rating"], reverse=True)[0]

    return best_cake


# Step 5: Simulate user query
user_budget = 500
result = find_best_cake(user_budget)

# Step 6: Output
if isinstance(result, dict):
    print("\n🎂 Best Cake Found:")
    print(f"Name   : {result['name']}")
    print(f"Price  : ₹{result['price']}")
    print(f"Rating : ⭐ {result['rating']}")
    print(f"Shop   : {result['shop']}")
else:
    print(result)

🔍 Searching for chocolate cakes under ₹ 500

🎂 Best Cake Found:
Name   : Premium Chocolate Cake
Price  : ₹500
Rating : ⭐ 4.8
Shop   : Delight Bakery


senario 3:
Scenario: The University’s AI Plagiarism Agent
Imagine a university has deployed an AI-powered plagiarism detection agent named PlagiarismLLM.
Its job is to monitor student essay submissions in real time and decide whether they are original or potentially plagiarized.

Step 1: Thinking (Reasoning)
Whenever an essay is submitted, PlagiarismLLM first analyzes the text:
- Example: “Essay contains large verbatim passages from online sources”
- PlagiarismLLM thinks: “Analyzing essay: multiple verbatim passages detected.”

Step 2: Planning (Choosing an Action)
Based on the thought, PlagiarismLLM decides which tool to use:
- If the essay has long verbatim matches or suspicious similarity scores, it plans to flag as potential plagiarism.
- Otherwise, it plans to mark as original work.

Step 3: Acting (Running Tools)
PlagiarismLLM hands over the decision to PlagiarismTools, which executes the action:
- flag_plagiarism → returns “Potential Plagiarism Detected”
- mark_original → returns “Original Work”

Step 4: Answering (Final Decision)
Once a clear classification is reached, PlagiarismLLM announces the decision:
- “Decision: Potential Plagiarism Detected”
- “Decision: Original Work”

In [1]:
class PlagiarismLLM:
    def think(self, input_text):
        # Simulate reasoning based on the input text
        if "verbatim passages" in input_text.lower() or "suspicious similarity" in input_text.lower():
            return f"Analyzing essay: {input_text}"
        else:
            return f"Analyzing essay: {input_text}"

    def plan(self, thought):
        # Decide which tool to use based on the thought
        if "verbatim passages" in thought.lower() or "suspicious similarity" in thought.lower():
            return "flag_plagiarism"
        else:
            return "mark_original"

    def answer(self, obs):
        # Final answer
        return f"Decision: {obs}"

class PlagiarismTools:
    def run(self, action):
        # Execute plagiarism detection actions
        if action == "flag_plagiarism":
            return "Potential Plagiarism Detected"
        elif action == "mark_original":
            return "Original Work"
        else:
            return "Unknown Action"

def is_final_plagiarism_decision(obs):
    # Stop once we have a clear classification
    return "Plagiarism Detected" in obs or "Original Work" in obs

# Instantiate components
plagiarism_llm = PlagiarismLLM()
plagiarism_tools = PlagiarismTools()

def react_plagiarism_agent(essay_analysis):
    print(f"Initial Analysis: {essay_analysis}")
    thought = plagiarism_llm.think(essay_analysis)
    print(f"Thought: {thought}")
    action = plagiarism_llm.plan(thought)
    print(f"Action: {action}")
    obs = plagiarism_tools.run(action)
    print(f"Observation: {obs}")

    # Loop until final decision (simplified for this example, usually involves more iterations)
    while not is_final_plagiarism_decision(obs):
        # In a real ReAct agent, the next thought would be based on the previous observation
        # For this toy example, we'll assume a single step leads to a final decision
        print("\n(Simulating another ReAct cycle if needed - for this example, one cycle is usually enough)")
        thought = plagiarism_llm.think(obs)
        action = plagiarism_llm.plan(thought)
        obs = plagiarism_tools.run(action)
        print(f"Observation after further processing: {obs}")

    return plagiarism_llm.answer(obs)

# 🚀 Try it out with different scenarios
print("\n--- Scenario 1: Essay with verbatim passages ---")
print(react_plagiarism_agent("Essay contains large verbatim passages from online sources"))

print("\n--- Scenario 2: Original student work ---")
print(react_plagiarism_agent("Well-researched essay with proper citations, no suspicious similarities."))

print("\n--- Scenario 3: Suspicious similarity scores ---")
print(react_plagiarism_agent("Submission has suspicious similarity scores compared to previous submissions."))


--- Scenario 1: Essay with verbatim passages ---
Initial Analysis: Essay contains large verbatim passages from online sources
Thought: Analyzing essay: Essay contains large verbatim passages from online sources
Action: flag_plagiarism
Observation: Potential Plagiarism Detected
Decision: Potential Plagiarism Detected

--- Scenario 2: Original student work ---
Initial Analysis: Well-researched essay with proper citations, no suspicious similarities.
Thought: Analyzing essay: Well-researched essay with proper citations, no suspicious similarities.
Action: mark_original
Observation: Original Work
Decision: Original Work

--- Scenario 3: Suspicious similarity scores ---
Initial Analysis: Submission has suspicious similarity scores compared to previous submissions.
Thought: Analyzing essay: Submission has suspicious similarity scores compared to previous submissions.
Action: flag_plagiarism
Observation: Potential Plagiarism Detected
Decision: Potential Plagiarism Detected


senario 4
###using gemeni api gemeni 2 fast



In [7]:
# use gemeni api key by the name of "google_api_key" in the secreate colab folder , dont change the code too much just update the api appliction
import requests
import google.generativeai as genai
from google.colab import userdata

# Retrieve API key from Colab secrets
GOOGLE_API_KEY=userdata.get('google_api_key')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini API model
gemini_model = genai.GenerativeModel('gemini-2.5-flash') # Updated to gemini-2.5-flash

def query_llm(prompt: str):
    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error querying Gemini API: {e}"

def flag_suspicious(transaction):
    return "Suspicious Transaction"

def mark_normal(transaction):
    return "Normal Transaction"

def fraud_agent(transaction: str):
    # Step 1: Thinking (Reasoning)
    thought = query_llm(f"Analyze this transaction: {transaction}. Provide a concise analysis.")
    print("LLM Thought:", thought)

    # Step 2: Planning (Choosing an Action)
    # The planning logic remains based on keywords in the transaction
    if "overseas" in transaction.lower() or "multiple deposits" in transaction.lower():
        action = flag_suspicious
    else:
        action = mark_normal

    # Step 3: Acting (Running Tools)
    result = action(transaction)

    # Step 4: Answering (Final Decision)
    final_answer = query_llm(f"Given the analysis: '{thought}' and the action taken: '{result}', formulate a final decision for the user.")
    return final_answer

print(fraud_agent("Payment to overseas account"))
print(fraud_agent("Salary credited to employee account"))
print(fraud_agent("Multiple small deposits from different sources"))

LLM Thought: A "Payment to overseas account" is a **cross-border financial transaction** involving the transfer of funds from a sender in one country to a recipient in another.

**Concise Analysis:**

1.  **Purpose:** Highly diverse, ranging from personal remittances (family support, online purchases, investments) to business-related payments (supplier invoices, salaries, inter-company transfers, service fees).
2.  **Method:** Typically executed via bank wire transfers (SWIFT, SEPA), international payment platforms (e.g., Wise, PayPal), or specialized money transfer services.
3.  **Currency & Cost:** Often involves foreign exchange (FX) conversion, incurring transfer fees, FX margins, and potentially intermediary bank charges.
4.  **Regulatory & Risk:** Subject to stringent international regulations, including Anti-Money Laundering (AML), Counter-Terrorism Financing (CTF), and sanctions. Carries higher risks of delays, errors, fraud, FX volatility, and increased scrutiny by financial a

senario 5:
🏥 Scenario: The Hospital’s AI Appointment Agent
Imagine a hospital has deployed an AI-powered agent named MediLLM.
Its job is to monitor patient appointment requests and decide whether the requested slot is available or needs rescheduling.

Step 1 — Thinking (Reasoning)
Whenever a patient request comes in, MediLLM first analyzes the details:
- Example: “Request for Dr. Sharma at 10 AM tomorrow”
- MediLLM thinks: “Analyzing appointment request: Dr. Sharma, 10 AM tomorrow.”

Step 2 — Planning (Choosing an Action)
Based on the thought, MediLLM decides which tool to use:
- If the requested slot is free, it plans to confirm appointment.
- If the slot is already booked, it plans to suggest rescheduling.

Step 3 — Acting (Running Tools)
MediLLM hands over the decision to AppointmentTools, which executes the action:
- confirm_slot → returns “Appointment Confirmed”
- suggest_reschedule → returns “Slot Unavailable, Please Reschedule”

Step 4 — Answering (Final Decision)
Once a clear classification is reached, MediLLM announces the decision:
- “Decision: Appointment Confirmed”
- “Decision: Slot Unavailable, Please Reschedule”

In [8]:
class MediLLM:
    def think(self, request_text):
        # Simulate reasoning based on the request text
        return f"Analyzing appointment request: {request_text}"

    def plan(self, thought):
        # Decide which tool to use based on the thought
        # For simplicity, let's assume 'tomorrow' and '10 AM' is generally busy
        # In a real system, this would involve checking a calendar/database
        if "10 AM tomorrow" in thought.lower() or "fully booked" in thought.lower():
            return "suggest_reschedule"
        else:
            return "confirm_slot"

    def answer(self, obs):
        # Final answer
        return f"Decision: {obs}"

class AppointmentTools:
    def run(self, action):
        # Execute appointment actions
        if action == "confirm_slot":
            return "Appointment Confirmed"
        elif action == "suggest_reschedule":
            return "Slot Unavailable, Please Reschedule"
        else:
            return "Unknown Action"

def is_final_appointment_decision(obs):
    # Stop once we have a clear classification
    return "Confirmed" in obs or "Unavailable" in obs or "Reschedule" in obs

# Instantiate components
medillm = MediLLM()
appointment_tools = AppointmentTools()

def react_appointment_agent(appointment_request):
    print(f"Initial Request: {appointment_request}")
    thought = medillm.think(appointment_request)
    print(f"Thought: {thought}")
    action = medillm.plan(thought)
    print(f"Action: {action}")
    obs = appointment_tools.run(action)
    print(f"Observation: {obs}")

    # In a real ReAct agent, the next thought would be based on the previous observation
    # For this toy example, we'll assume a single step leads to a final decision
    # The loop here is more for demonstration of the ReAct cycle, even if it's one iteration.
    while not is_final_appointment_decision(obs):
        print("\n(Simulating another ReAct cycle if needed - for this example, one cycle is usually enough)")
        thought = medillm.think(obs) # The LLM might re-evaluate based on the observation
        action = medillm.plan(thought)
        obs = appointment_tools.run(action)
        print(f"Observation after further processing: {obs}")

    return medillm.answer(obs)

# 🚀 Try it out with different scenarios
print("\n--- Scenario 1: Request for Dr. Sharma at 10 AM tomorrow ---")
print(react_appointment_agent("Request for Dr. Sharma at 10 AM tomorrow"))

print("\n--- Scenario 2: Request for Dr. Smith at 2 PM today ---")
print(react_appointment_agent("Request for Dr. Smith at 2 PM today"))

print("\n--- Scenario 3: Request for a fully booked slot ---")
print(react_appointment_agent("Request for Dr. Lee for a slot that is fully booked"))


--- Scenario 1: Request for Dr. Sharma at 10 AM tomorrow ---
Initial Request: Request for Dr. Sharma at 10 AM tomorrow
Thought: Analyzing appointment request: Request for Dr. Sharma at 10 AM tomorrow
Action: confirm_slot
Observation: Appointment Confirmed
Decision: Appointment Confirmed

--- Scenario 2: Request for Dr. Smith at 2 PM today ---
Initial Request: Request for Dr. Smith at 2 PM today
Thought: Analyzing appointment request: Request for Dr. Smith at 2 PM today
Action: confirm_slot
Observation: Appointment Confirmed
Decision: Appointment Confirmed

--- Scenario 3: Request for a fully booked slot ---
Initial Request: Request for Dr. Lee for a slot that is fully booked
Thought: Analyzing appointment request: Request for Dr. Lee for a slot that is fully booked
Action: suggest_reschedule
Observation: Slot Unavailable, Please Reschedule
Decision: Slot Unavailable, Please Reschedule
